In [24]:
# impoerting the tools
import pandas as pd
import numpy as np

# importing the dataset
from sklearn.datasets import load_breast_cancer

# importing the ml tools

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

In [25]:
# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target # 0 = malignant, 1 = benign

# Train-test split (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data: {X_train.shape[0]} patients")
print(f"Testing data: {X_test.shape[0]} patients")

# Note: We are NOT manually scaling the data here.
# We will use Scikit-Learn 'Pipelines' to handle scaling automatically!

Training data: 455 patients
Testing data: 114 patients


In [26]:
# Create pipelines that bundle the Scaler and the Model together
pipeline_lr = Pipeline([('scaler', StandardScaler()),
                         ('model', LogisticRegression())])

pipeline_dt = Pipeline([('scaler', StandardScaler()), ('model',
                                                        DecisionTreeClassifier(random_state=42))])

pipeline_rf = Pipeline([('scaler', StandardScaler()),
                         ('model', RandomForestClassifier(random_state=42))])

# Train the models using .fit()
pipeline_lr.fit(X_train, y_train)
pipeline_dt.fit(X_train, y_train)
pipeline_rf.fit(X_train, y_train)

print("All 3 models trained successfully using Pipelines!")

All 3 models trained successfully using Pipelines!


In [27]:
# --- Evaluating Logistic Regression ---
print("==== LOGISTIC REGRESSION ====")
print(f"Train Accuracy: {pipeline_lr.score(X_train, y_train):.4f}")
print(f"Test Accuracy:  {pipeline_lr.score(X_test, y_test):.4f}")

y_pred_lr = pipeline_lr.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))
print("-" * 50)

==== LOGISTIC REGRESSION ====
Train Accuracy: 0.9868
Test Accuracy:  0.9737

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.95      0.96        43
           1       0.97      0.99      0.98        71

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114

--------------------------------------------------


In [28]:
# --- Evaluating Decision Tree ---
print("==== DECISION TREE ====")
print(f"Train Accuracy: {pipeline_dt.score(X_train, y_train):.4f}")
print(f"Test Accuracy:  {pipeline_dt.score(X_test, y_test):.4f}")

y_pred_dt = pipeline_dt.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))
print("-" * 50)

==== DECISION TREE ====
Train Accuracy: 1.0000
Test Accuracy:  0.9474

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.93      0.93        43
           1       0.96      0.96      0.96        71

    accuracy                           0.95       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114

--------------------------------------------------


In [29]:
# --- Evaluating Random Forest ---
print("==== RANDOM FOREST ====")
print(f"Train Accuracy: {pipeline_rf.score(X_train, y_train):.4f}")
print(f"Test Accuracy:  {pipeline_rf.score(X_test, y_test):.4f}")

y_pred_rf = pipeline_rf.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))
print("-" * 50)

==== RANDOM FOREST ====
Train Accuracy: 1.0000
Test Accuracy:  0.9649

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.93      0.95        43
           1       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114

--------------------------------------------------


### 4. Building a Metric Table for Comparison
Instead of scrolling through logs, professionals build **Metric Tables** to see everything at once. We'll specifically look at Train vs Test accuracy to spot **Overfitting**.

***Occam's Razor Principle:*** *If a simple model (Logistic Regression) performs almost identically to a complex model (Random Forest), we prefer the simpler one because it uses less memory and is less prone to overfitting!*

In [30]:
# Calculate scores manually
train_lr, test_lr = pipeline_lr.score(X_train, y_train), pipeline_lr.score(X_test, y_test)
train_dt, test_dt = pipeline_dt.score(X_train, y_train), pipeline_dt.score(X_test, y_test)
train_rf, test_rf = pipeline_rf.score(X_train, y_train), pipeline_rf.score(X_test, y_test)

# Collate into a simple list of dictionaries
results = [
    {"Model": "Logistic Regression", "Train Accuracy": train_lr, "Test Accuracy": test_lr, "Gap": train_lr - test_lr},
    {"Model": "Decision Tree",       "Train Accuracy": train_dt, "Test Accuracy": test_dt, "Gap": train_dt - test_dt},
    {"Model": "Random Forest",       "Train Accuracy": train_rf, "Test Accuracy": test_rf, "Gap": train_rf - test_rf}
]

df_results = pd.DataFrame(results).set_index("Model")
display(df_results.round(4))

print("\n🔍 OBSERVATION:")
print("Notice how Decision Tree got a 1.0 (100%) Train Accuracy but lower Test Accuracy. It memorized the data! (Overfitting)")
print("Logistic Regression is the clear winner here. It is simple, fast, and highly accurate. No complex tuning needed!")

,Train Accuracy,Test Accuracy,Gap
Model,,,
Logistic Regression,0.9868,0.9737,0.0131
Decision Tree,1.0000,0.9474,0.0526
Random Forest,1.0000,0.9649,0.0351



🔍 OBSERVATION:
Notice how Decision Tree got a 1.0 (100%) Train Accuracy but lower Test Accuracy. It memorized the data! (Overfitting)
Logistic Regression is the clear winner here. It is simple, fast, and highly accurate. No complex tuning needed!


### 5. Understanding Overfitting vs. Underfitting (Live Demo)
Let's explicitly see how model complexity affects our performance by changing `max_depth` (Decision Tree) and `n_estimators` (Random Forest).

In [31]:
print("--- Decision Tree: Varying 'max_depth' ---")
# depth=1 means a very shallow tree (Underfitting)
# depth=None means the tree grows infinitely until it memorizes everything (Overfitting)
for depth in [1, 3, None]:
    dt = Pipeline([('scaler', StandardScaler()), ('model', DecisionTreeClassifier(max_depth=depth, random_state=42))])
    dt.fit(X_train, y_train)
    print(f"Depth {str(depth):>4} | Train Acc: {dt.score(X_train, y_train):.4f} | Test Acc: {dt.score(X_test, y_test):.4f}")

print("\n--- Random Forest: Varying 'n_estimators' ---")
# n=1 is literally just a single tree (Underfitting/Weak)
# n=100 is a full forest (Strong, but might memorize train data slightly)
for n in [1, 10, 100]:
    rf = Pipeline([('scaler', StandardScaler()), ('model', RandomForestClassifier(n_estimators=n, random_state=42))])
    rf.fit(X_train, y_train)
    print(f"Trees {str(n):>4} | Train Acc: {rf.score(X_train, y_train):.4f} | Test Acc: {rf.score(X_test, y_test):.4f}")

--- Decision Tree: Varying 'max_depth' ---
Depth    1 | Train Acc: 0.9209 | Test Acc: 0.8947
Depth    3 | Train Acc: 0.9780 | Test Acc: 0.9474
Depth None | Train Acc: 1.0000 | Test Acc: 0.9474

--- Random Forest: Varying 'n_estimators' ---
Trees    1 | Train Acc: 0.9582 | Test Acc: 0.9386
Trees   10 | Train Acc: 0.9978 | Test Acc: 0.9561
Trees  100 | Train Acc: 1.0000 | Test Acc: 0.9649
